# Semaine 9 — Jour 2 : Embeddings, similarité et compatibilité

Calculer un encodage vectoriel déterministe et une similarité cosinus.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. Les modèles et services optionnels ajoutent un temps d’installation distinct.

Prérequis : Python 3.11+, JSON, notions de vecteurs et de persistance. Le parcours obligatoire utilise uniquement la bibliothèque standard et ne contacte aucun service externe.

Lire [les objectifs](learning_objectives.md), [le chapitre](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week09/S9_J2_embeddings.ipynb` et `notebooks/week09/teacher/S9_J2_embeddings_teacher.ipynb`.

Critère de réussite : Refuser le mélange de modèles, dimensions et normalisations incompatibles.

Les calculs de retrieval et de persistance sont exécutables; le corpus est fictif. L’encodeur de base est lexical et la réponse finale est extractive. Les extensions neuronales sont séparées et n’ont pas été exécutées ici.


# Objectifs mesurables

- Calculer un encodage vectoriel déterministe et une similarité cosinus.
- Distinguer projection lexicale et embedding neuronal sémantique.
- Refuser le mélange de modèles, dimensions et normalisations incompatibles.

## Preuves attendues

Une sortie reproductible, un contrôle d’invariant et une limite du mécanisme correctement expliquée.


# S9 J2 — Embeddings, similarité et compatibilité


## 1. Représenter un texte par des nombres
Un embedding transforme une entrée en un vecteur. Son intérêt dépend de ce que sa géométrie encode. Deux textes proches selon le vecteur peuvent partager des mots, un thème, une intention ou seulement une collision de représentation. La dimension n’est pas un score de qualité et un cosinus élevé ne devient pas une probabilité de réponse correcte.

Le lab obligatoire implémente HashEmbedding : une projection signée des comptes de mots vers 128 dimensions. C’est une baseline lexicale calculée réellement, sans valeurs préfabriquées. Elle n’apprend rien et ne comprend pas les synonymes. La nommer « recherche sémantique neuronale » serait incorrect.

## 2. Projection signée et normalisation
La fonction tokens normalise casse et accents, retire une liste fixe de mots courants et conserve des termes alphanumériques. Chaque terme est associé par SHA-256 à une case et à un signe; son compte est ajouté à la case. La norme L2 sert ensuite à normaliser le vecteur. Un texte sans termes utiles produit le vecteur nul.

SHA-256 garantit ici une projection déterministe entre processus, contrairement à hash() de Python dont le comportement n’est pas destiné à définir un index stable. Cela ne supprime pas les collisions : plusieurs termes peuvent atteindre la même dimension et leurs signes peuvent se compenser. Une faible dimension rend ce problème plus visible.

## 3. Similarité cosinus
Le cosinus est le produit scalaire divisé par le produit des normes. Pour deux vecteurs non nuls normalisés, il coïncide avec le produit scalaire. Le lab retourne zéro lorsqu’un vecteur est nul et refuse les dimensions différentes ou les valeurs non finies.

« Kafka paiement » et « paiement Kafka » possèdent le même sac de mots. Leur cosinus vaut environ un, mais l’ordre des mots a disparu. Une négation peut modifier fortement le sens sans changer suffisamment la représentation. C’est une limite du modèle lexical, pas un bug du calcul de cosinus.

## 4. Passer à un modèle appris
Un bi-encoder neuronal calcule séparément les vecteurs de requête et de document. Certains modèles utilisent des prompts ou tâches distincts pour ces deux côtés. L’adaptateur optionnel applique encode_query et encode_document et normalise leurs sorties. Il exige des poids déjà présents localement; aucun téléchargement n’est déclenché par le code.

Choisir un modèle pour sa langue, sa tâche et sa contrainte d’exploitation. Tester les paraphrases, identifiants techniques et cas négatifs. La requête q8 emploie une paraphrase éloignée du vocabulaire du corpus : elle expose une faiblesse de la baseline plutôt que de masquer les échecs derrière des questions trop faciles. Les modèles neuronaux et leur qualité ne sont pas validés dans cet environnement.

## 5. Compatibilité d’index
Le store conserve une empreinte du modèle et sa dimension. Ouvrir une base hashbow en 64 dimensions avec un encodeur de 128 dimensions est refusé. Changer de modèle avec la même dimension doit aussi être refusé : la signification des axes peut être différente. Une comparaison de vecteurs de longueur égale ne suffit pas.

Versionner aussi le tokenizer, les prompts d’encodage et la normalisation. L’adaptateur neuronal calcule une empreinte des artefacts locaux. Pour une migration, créer un index distinct, le reconstruire et comparer les résultats avant bascule. Mélanger progressivement deux espaces dans une même collection rend les scores incohérents.

## 6. Expérience du jour
Comparer vecteurs identiques, vecteur nul, ordre des mots inversé et dimension très faible. Distinguer le résultat du calcul et son interprétation métier. Le cours de S1 expliquait le concept; ici l’enjeu est son contrat d’indexation et la migration de ses artefacts.

## À retenir
La géométrie d’un embedding dépend de sa construction. Un espace compatible et une expérience de pertinence comptent davantage qu’une grande dimension ou un score isolé.


## Diagramme du jour

```mermaid
flowchart TD
 A["Texte et tokenizer"] --> B["Encodeur versionné"]
 B --> C["Vecteur et dimension"]
 C --> D["Normalisation"]
 D --> E["Cosinus"]
 F["Empreinte d’index"] --> G["Contrôle de compatibilité"]
 C --> G
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

Les cellules s’exécutent hors ligne depuis l’arbre du dépôt.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week09/knowledge.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
import math
from examples.week09.knowledge import HashEmbedding, cosine, ContractError
encoder = HashEmbedding(128)
a = encoder.encode("Kafka paiement")
b = encoder.encode("paiement Kafka")
assert a == b and math.isclose(cosine(a, b), 1.0)
assert math.isclose(cosine([1, 0], [1, 1]), 1 / math.sqrt(2))
assert cosine([0, 0], [1, 1]) == 0
print(encoder.fingerprint)

# Lab J2 — Embeddings, similarité et compatibilité

Depuis la racine du dépôt :

```bash
python3 book/week09/day02/labs/run.py
python3 -m examples.week09.demo --day 2
```

Résultat attendu : **128 dimensions, même sac de mots : cosinus proche de 1**.

## Manipulations

1. Reproduire la sortie et retrouver le calcul dans `examples/week09/knowledge.py`.
2. Réaliser l’exercice 2 avec une copie des données, sans modifier le corpus distribué.
3. Vérifier l’invariant étudié et expliquer une limite.

Le corpus se trouve dans `examples/week09/corpus`. Les tests automatisés sont sous `tests/week09`. [QUICKSTART](../../../../docs/week09/QUICKSTART.md) décrit les commandes et les codes retour.

Les modèles locaux et pgvector sont optionnels : lire `examples/week09/optional/README.md` depuis la racine du dépôt. Ils ne sont pas nécessaires pour finir cette journée et ne sont pas présentés comme validés dans cet environnement.


# Exercices

Conserver votre première réponse avant de lire les solutions.

## Exercice 1 — Calculer le cosinus

Calculer le cosinus de [1,0] et [1,1], puis expliquer le résultat pour un vecteur nul.

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 2 — Diagnostiquer les synonymes

Pourquoi hashbow peut-il échouer sur une paraphrase sans mots communs ?

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 3 — Préparer une migration

Deux modèles ont chacun 384 dimensions. Peut-on conserver leurs vecteurs dans le même espace de recherche ?

Livrable : réponse argumentée avec calcul, test ou observation.


# Entretien

Répondre en deux minutes avec un exemple concret et une limite.

## Question 1

Un cosinus de 0,9 signifie-t-il 90 % de confiance ?

## Question 2

Pourquoi distinguer encode_query et encode_document ?


# Challenge

Montrer que l’ordre des mots est perdu et que deux espaces de dimensions différentes sont refusés. Décrire une expérience qui testerait réellement les synonymes avec l’adaptateur neuronal.

## Critères de réussite

Reproductibilité, provenance des données, invariant vérifié et limite expliquée sans exagérer les garanties.


# Références — J2

Sources primaires consultées le 20 octobre 2026. Les algorithmes et contrats du lab sont explicités dans le chapitre; le corpus est une création fictive du cours.

- [pgvector : documentation officielle](https://github.com/pgvector/pgvector) — stockage, distances, recherche exacte/ANN et filtres.
- [Sentence Transformers : retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) — architecture de récupération en deux étapes.
- [SentenceTransformer : API](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html) — encode_query, encode_document et modèles locaux.
- [CrossEncoder : API](https://www.sbert.net/docs/package_reference/cross_encoder/model.html) — scoring conjoint de paires.
- [Cormack, Clarke et Buettcher, SIGIR 2009 : Reciprocal Rank Fusion](https://research.google/pubs/reciprocal-rank-fusion-outperforms-condorcet-and-individual-rank-learning-methods/) — origine de la méthode de fusion des rangs.
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) — persistance locale et transactions.

Lire en priorité les références du thème du jour. Les extensions évoluent; leur environnement réel doit être verrouillé et testé avant utilisation. Le paquet n’affirme pas avoir exécuté de modèle neuronal ou de serveur pgvector.
